# 4. 데이터 결합
## 4.1 데이터 불러오기

In [38]:
import pandas as pd

In [39]:
bike=pd.read_csv("bike_sampled_preprocess.csv", encoding="cp949", low_memory=False)
weather = pd.read_csv("weather_bike_preprocess.csv", encoding="cp949", low_memory=False)


## 4.2 bike 기준으로 결합하기

In [40]:
# 1) 문자열 양쪽 공백 제거
bike["datehour"] = bike["datehour"].astype(str).str.strip()
weather["datehour"] = weather["datehour"].astype(str).str.strip()

# 2) datetime 타입으로 강제 변환
bike["datehour"] = pd.to_datetime(bike["datehour"], errors='coerce')
weather["datehour"] = pd.to_datetime(weather["datehour"], errors='coerce')

# 3) 정각(hour) 단위로 통일 (초/분 제거)
bike["datehour"] = bike["datehour"].dt.floor("H")
weather["datehour"] = weather["datehour"].dt.floor("H")


C:\Users\sooyu\AppData\Local\Temp\ipykernel_20552\2494725857.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  bike["datehour"] = bike["datehour"].dt.floor("H")
C:\Users\sooyu\AppData\Local\Temp\ipykernel_20552\2494725857.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather["datehour"] = weather["datehour"].dt.floor("H")


In [41]:
bike_weather = bike.merge(weather, on="datehour", how="left")

In [42]:
bike_weather.head()

,rent_datetime,rent_station_id,rent_station_name,return_datetime,return_station_id,return_station_name,use_time_min,use_distance_m,birth_year,gender,...,wind,humid,sunshine,solar,cloud,visibility,weekday_y,holiday,count,rain_flag
0,2024-09-24 16:27:54,750.0,연의근린공원 건너편,2024-09-24 17:23:01,4538.0,양천공영차고지,55,2369.29,2001.0,M,...,2.1,41.0,1.0,1.98,0.0,3327.0,Tuesday,0.0,10619.0,0.0
1,2024-09-10 15:22:48,215.0,여의도고교 앞,2024-09-10 15:50:43,3214.0,영등포 푸르지오 아파트 A,27,3623.98,1966.0,F,...,2.8,54.0,0.9,2.27,3.0,1587.0,Tuesday,0.0,5360.0,0.0
2,2024-09-28 20:40:17,1906.0,신도림역 1번 출구 앞,2024-09-28 21:23:18,2183.0,동방1교,43,7886.93,1977.0,M,...,3.4,58.0,0.0,0.00,2.0,3963.0,Saturday,0.0,10214.0,0.0
3,2024-09-27 13:32:37,1451.0,중랑세무서,2024-09-27 13:37:40,1404.0,동일로 지하차도,5,718.45,1984.0,M,...,2.5,62.0,1.0,2.98,3.0,4184.0,Friday,0.0,6257.0,0.0
4,2024-09-28 20:28:00,1022.0,길동 사거리(초소앞),2024-09-28 20:42:18,2622.0,올림픽공원역 3번출구,14,3860.00,2000.0,M,...,3.4,58.0,0.0,0.00,2.0,3963.0,Saturday,0.0,10214.0,0.0


In [43]:
bike_weather.isnull().sum()

rent_datetime            0
rent_station_id          0
rent_station_name        0
return_datetime          0
return_station_id        0
return_station_name      0
use_time_min             0
use_distance_m           0
birth_year               0
gender                   0
user_type                0
rent_station_uid         0
return_station_uid       0
bike_type                0
datehour                 0
actual_use_time          0
weekday_x                0
weekday_name             0
weather_datetime       263
temp                   263
rain                   263
wind                   263
humid                  263
sunshine               263
solar                  263
cloud                  263
visibility             263
weekday_y              263
holiday                263
count                  263
rain_flag              263
dtype: int64

## 4.3 시간 순으로 정렬

In [44]:
bike_weather = bike_weather.sort_values("datehour").reset_index(drop=True)

## 4.4 결측값이 생긴 열들 처리

In [45]:
#중복된 열과 불필요한 열들은 삭제
bike_weather =bike_weather.drop(columns=["weather_datetime"])
bike_weather = bike_weather.drop(columns=["holiday"])
bike_weather = bike_weather.drop(columns=["count"])
bike_weather = bike_weather.drop(columns=["weekday_y"])


In [46]:
# 이전 값 채우기
# 날씨는 “시간에서 시간으로 점진적으로 변함”
weather_cols = ["temp","rain","wind","humid","sunshine","solar","cloud","visibility"]

bike_weather = bike_weather.sort_values("datehour")
bike_weather[weather_cols] = bike_weather[weather_cols].ffill()

In [47]:
# rain_flag 다시 계산
bike_weather["rain_flag"] = (bike_weather["rain"] > 0).astype(int)


## 4.5 열 정리(삭제, 재정렬, 추가)

In [48]:
cols_to_drop = [
    'rent_station_uid',
    'return_station_uid',
    'actual_use_time'
]

In [49]:
bike_weather =bike_weather.drop(columns=cols_to_drop, errors="ignore")


In [50]:
bike_weather.columns

Index(['rent_datetime', 'rent_station_id', 'rent_station_name',
       'return_datetime', 'return_station_id', 'return_station_name',
       'use_time_min', 'use_distance_m', 'birth_year', 'gender', 'user_type',
       'bike_type', 'datehour', 'weekday_x', 'weekday_name', 'temp', 'rain',
       'wind', 'humid', 'sunshine', 'solar', 'cloud', 'visibility',
       'rain_flag'],
      dtype='object')

In [51]:
new_order = [
    'rent_datetime', 'datehour',
    'rent_station_id', 'rent_station_name',
    'return_datetime', 'return_station_id', 'return_station_name',

    'birth_year', 'gender', 'user_type', 'bike_type',

    'use_time_min', 'use_distance_m', 'weekday_name',

    'temp', 'rain', 'rain_flag',
    'wind', 'humid', 'sunshine', 'solar', 'cloud',
    'visibility'
]

bike_weather = bike_weather[new_order]

In [52]:
# 연령대 열 추가
bike_weather["birth_year"] = pd.to_numeric(bike_weather["birth_year"], errors="coerce")
bike_weather["age"] = 2024 - bike_weather["birth_year"]
bike_weather["age_group"] = pd.cut(bike_weather["age"], bins=[0,20,30,40,50,60,100],
                         labels=["10대","20대","30대","40대","50대","60대+"])


In [56]:
bike_weather["age_group"] = bike_weather["age_group"].cat.add_categories(["미상"])
bike_weather["age_group"] = bike_weather["age_group"].fillna("미상")

In [57]:
bike_weather.to_csv("bike_weather_merged.csv", index=False, encoding="cp949")